
# TASK 2.2 – DATA DICTIONARY & KPI

## Mục tiêu
Xây dựng Data Dictionary, định nghĩa Business KPI,
Technical KPI và kiểm tra chất lượng dữ liệu Airbnb Singapore.

Dữ liệu được đọc từ bốn nguồn:
- Listings
- Calendar
- Reviews
- Neighbourhoods

Notebook không chỉnh sửa dữ liệu raw.


In [1]:

from pathlib import Path
import pandas as pd
from IPython.display import display

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)


from pathlib import Path
import pandas as pd
from IPython.display import display

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

# Tìm thư mục gốc của project từ thư mục hiện tại
current_path = Path.cwd()

candidates = [current_path] + list(current_path.parents)

ROOT = next(
    (
        path for path in candidates
        if (
            (path / "data" / "raw").exists()
            and (path / "docs" / "data_dictionary").exists()
            and (path / "outputs" / "task_2_2").exists()
        )
    ),
    None
)

if ROOT is None:
    raise FileNotFoundError(
        "Không tìm thấy thư mục gốc Singapore_Airbnb."
    )

DD_DIR = ROOT / "docs" / "data_dictionary"
KPI_DIR = ROOT / "docs" / "kpi_definitions"
OUT_DIR = ROOT / "outputs" / "task_2_2"

print("Thư mục dự án:", ROOT)
print("Data Dictionary:", DD_DIR)
print("KPI Definitions:", KPI_DIR)
print("Kết quả kiểm tra:", OUT_DIR)


print("Thư mục dự án:", ROOT)
print("Data Dictionary:", DD_DIR)
print("KPI Definitions:", KPI_DIR)
print("Kết quả kiểm tra:", OUT_DIR)


Thư mục dự án: E:\NKDL\Singapore_Airbnb
Data Dictionary: E:\NKDL\Singapore_Airbnb\docs\data_dictionary
KPI Definitions: E:\NKDL\Singapore_Airbnb\docs\kpi_definitions
Kết quả kiểm tra: E:\NKDL\Singapore_Airbnb\outputs\task_2_2
Thư mục dự án: E:\NKDL\Singapore_Airbnb
Data Dictionary: E:\NKDL\Singapore_Airbnb\docs\data_dictionary
KPI Definitions: E:\NKDL\Singapore_Airbnb\docs\kpi_definitions
Kết quả kiểm tra: E:\NKDL\Singapore_Airbnb\outputs\task_2_2


## 1. Tổng quan bốn nguồn dữ liệu

In [2]:
source_profile = pd.read_csv(DD_DIR / 'source_profile_for_dictionary.csv')
print('Thông tin nguồn dữ liệu và snapshot:')
display(source_profile)

sources = ['listings', 'calendar', 'reviews', 'neighbourhoods']
profiles = []
for source in sources:
    df = pd.read_csv(DD_DIR / f'data_dictionary_{source}.csv')
    profiles.append({
        'Nguồn': source.title(),
        'Số trường': len(df),
        'Số trường có mô tả': int(df['business_meaning'].notna().sum()),
        'Số dòng nguồn': int(df['row_count_source'].iloc[0])
    })
display(pd.DataFrame(profiles))

Thông tin nguồn dữ liệu và snapshot:


,source_name,source_file,snapshot_date,row_count,column_count,grain
0,Listings,listings.csv.gz,2026-09-23,3208,90,Một listing tại một snapshot
1,Calendar,calendar.csv.gz,2026-09-23,1170950,5,Một listing tại một ngày lịch và một snapshot
2,Reviews,reviews.csv.gz,2026-09-23,46000,6,Một bản ghi đánh giá
3,Neighbourhoods,neighbourhoods.csv,2026-09-23,55,2,Một khu vực


,Nguồn,Số trường,Số trường có mô tả,Số dòng nguồn
0,Listings,90,90,3208
1,Calendar,5,5,1170950
2,Reviews,6,6,46000
3,Neighbourhoods,2,2,55


## 2. Data Dictionary – toàn bộ trường dữ liệu

In [3]:

dictionary = pd.read_csv(
    DD_DIR / "data_dictionary_all_sources.csv"
)

print("Tổng số trường:", len(dictionary))
print("Số nguồn:", dictionary["source_name"].nunique())

display(dictionary)


Tổng số trường: 103
Số nguồn: 4


,source_name,source_file,column_name,business_meaning,initial_pandas_dtype,sample_value,row_count_source,null_count,null_rate_percent,nullable_observed,key_role,proposed_transformation,kpi_relevance,observed_range_or_values
0,Listings,listings.csv.gz,id,Mã định danh duy nhất của tin đăng Airbnb.,int64,71609,3208,0,0.0000,Không phát hiện NULL,Khóa định danh của listing; khi tích hợp nhiều...,Lưu kiểu số nguyên đủ miền giá trị (BIGINT); k...,Trường dùng trực tiếp/hỗ trợ KPI hoặc liên kết,NaN
1,Listings,listings.csv.gz,listing_url,Đường dẫn trang tin đăng trên Airbnb.,str,https://www.airbnb.com/rooms/71609,3208,0,0.0000,Không phát hiện NULL,Không phải khóa chính/khóa ngoại đã xác nhận.,Chuẩn hóa khoảng trắng và NULL; chọn VARCHAR/T...,Trường mô tả/hỗ trợ kiểm tra; chưa dùng trực t...,NaN
2,Listings,listings.csv.gz,scrape_id,Mã lần thu thập dữ liệu của nguồn.,int64,20260923225116,3208,0,0.0000,Không phát hiện NULL,Không phải khóa chính/khóa ngoại đã xác nhận.,Lưu kiểu số nguyên đủ miền giá trị (BIGINT); k...,Trường mô tả/hỗ trợ kiểm tra; chưa dùng trực t...,NaN
3,Listings,listings.csv.gz,last_scraped,Ngày tin đăng được thu thập lần cuối.,str,2026-09-26,3208,0,0.0000,Không phát hiện NULL,Không phải khóa chính/khóa ngoại đã xác nhận.,Chuẩn hóa thành DATE hoặc DATETIME; ghi nhận g...,Trường mô tả/hỗ trợ kiểm tra; chưa dùng trực t...,Ngày quan sát: 2026-09-25 đến 2026-09-26.
4,Listings,listings.csv.gz,source,Nguồn hoặc trạng thái xuất xứ của bản ghi theo...,str,previous scrape,3208,1470,45.8229,Có NULL,Không phải khóa chính/khóa ngoại đã xác nhận.,Chuẩn hóa khoảng trắng và NULL; chọn VARCHAR/T...,Trường mô tả/hỗ trợ kiểm tra; chưa dùng trực t...,NaN
5,Listings,listings.csv.gz,name,Tên hiển thị của tin đăng.,str,Ensuite Room (Room 1 & 2) near EXPO,3208,0,0.0000,Không phát hiện NULL,Không phải khóa chính/khóa ngoại đã xác nhận.,Chuẩn hóa khoảng trắng và NULL; chọn VARCHAR/T...,Trường mô tả/hỗ trợ kiểm tra; chưa dùng trực t...,NaN
6,Listings,listings.csv.gz,description,Nội dung mô tả chi tiết tin đăng.,str,For 3 rooms.Book room 1&2 and room 4,3208,21,0.6546,Có NULL,Không phải khóa chính/khóa ngoại đã xác nhận.,Giữ dạng TEXT; chuẩn hóa encoding; không dùng ...,Trường mô tả/hỗ trợ kiểm tra; chưa dùng trực t...,NaN
7,Listings,listings.csv.gz,neighborhood_overview,Mô tả tổng quan về khu vực xung quanh tin đăng.,float64,NaN,3208,3208,100.0000,Có NULL,Không phải khóa chính/khóa ngoại đã xác nhận.,Giữ dạng TEXT; chuẩn hóa encoding; không dùng ...,Trường mô tả/hỗ trợ kiểm tra; chưa dùng trực t...,NaN
8,Listings,listings.csv.gz,picture_url,Đường dẫn hình ảnh đại diện của tin đăng.,str,https://a0.muscache.com/pictures/24453191/3580...,3208,0,0.0000,Không phát hiện NULL,Không phải khóa chính/khóa ngoại đã xác nhận.,Chuẩn hóa khoảng trắng và NULL; chọn VARCHAR/T...,Trường mô tả/hỗ trợ kiểm tra; chưa dùng trực t...,NaN
9,Listings,listings.csv.gz,host_id,Mã định danh của chủ nhà.,int64,367042,3208,0,0.0000,Không phát hiện NULL,Không phải khóa chính/khóa ngoại đã xác nhận.,Lưu kiểu số nguyên đủ miền giá trị (BIGINT); k...,Trường mô tả/hỗ trợ kiểm tra; chưa dùng trực t...,NaN


## 3. Kiểm tra mô tả nghiệp vụ

In [4]:

meaning_audit = pd.read_csv(
    OUT_DIR / "dictionary_meaning_audit.csv"
)

print("Tổng hợp trạng thái mô tả:")

display(
    meaning_audit["audit_status"]
    .value_counts()
    .rename_axis("Trạng thái")
    .reset_index(name="Số trường")
)

print("Các trường cần rà soát:")

display(
    meaning_audit[
        meaning_audit["audit_status"]
        != "Đạt – có mô tả nghiệp vụ"
    ]
)


Tổng hợp trạng thái mô tả:


,Trạng thái,Số trường
0,Đạt – có mô tả nghiệp vụ,103


Các trường cần rà soát:


,source_name,column_name,business_meaning,audit_status


## 4. Kiểm tra Currency và Review Rating

In [5]:

source_checks = pd.read_csv(
    OUT_DIR / "source_validation_checks.csv"
)

display(source_checks)

print(
    "Lưu ý: Chỉ xác định tiền tệ SGD khi "
    "price_quote_raw.quote.currency ghi rõ SGD."
)

print(
    "Review Rating được kiểm tra trong miền từ 1 đến 5. "
    "Giá trị NULL không được xem là 0."
)


,check_id,noi_dung,so_ban_ghi_kiem_tra,ket_qua,trang_thai
0,SRC-01,Currency trong price_quote_raw.quote.currency,2890,"JSON báo giá có dữ liệu: 2,890; SGD: 2,855; JS...",Chỉ xác định SGD khi currency trong JSON bằng SGD
1,SRC-02,Miền giá trị review_scores_rating,1839,"Min: 1.0; Max: 5.0; Hợp lệ 1–5: 1,839; Ngoài m...",NULL không được tự động xem là điểm 0
2,SRC-03,Đối chiếu Listings.price với price_quote_price...,2725,"Có đủ hai giá trị để đối chiếu: 2,725; Khớp tr...",Chỉ đối chiếu các listing có currency SGD và đ...


Lưu ý: Chỉ xác định tiền tệ SGD khi price_quote_raw.quote.currency ghi rõ SGD.
Review Rating được kiểm tra trong miền từ 1 đến 5. Giá trị NULL không được xem là 0.


## 5. Business KPI và KPI hỗ trợ

In [6]:

business_kpi = pd.read_csv(
    KPI_DIR / "kpi_definitions.csv"
)

print("Số lượng Business KPI:", len(business_kpi))

display(business_kpi)


Số lượng Business KPI: 7


,kpi_id,kpi_name,business_definition,source_fields,formula,filters,numerator,denominator,unit,grain,interpretation,caveats
0,KPI-01,Số listing duy nhất theo snapshot,Số lượng mã listing phân biệt được ghi nhận tr...,Listings.id; Listings.scrape_id; Listings.last...,COUNT(DISTINCT id),Chỉ dùng một snapshot; loại id NULL; xác nhận ...,Số mã listing duy nhất hợp lệ.,Không áp dụng.,Listing,Một listing tại snapshot,"Quy mô tin đăng được ghi nhận tại snapshot, kh...",Không đếm dòng Calendar/Reviews; không gộp nhi...
1,KPI-02,Trung vị giá niêm yết mỗi đêm,Trung vị giá niêm yết mỗi đêm của các listing ...,Listings.id; Listings.price; Listings.price_qu...,"MEDIAN(price_clean), mỗi listing một lần; pric...","Một snapshot; price không NULL, parse được, hữ...",Không áp dụng cho median.,Không áp dụng; báo cáo coverage = listing có g...,SGD/đêm,Một listing tại snapshot,Một nửa listing có giá hợp lệ có giá không lớn...,Trong dữ liệu hiện tại có 2.855 báo giá ghi rõ...
2,KPI-03,Tỷ trọng loại phòng,Tỷ lệ listing thuộc từng room_type trên tổng l...,Listings.id; Listings.room_type; Listings.neig...,Số listing duy nhất thuộc room_type X / tổng l...,Giữ bộ lọc snapshot và khu vực; khi tính tỷ tr...,Số listing duy nhất thuộc room_type X trong ph...,Tổng listing duy nhất có room_type hợp lệ tron...,% listing,Một listing tại snapshot,Mô tả cơ cấu loại phòng trong tổng tin đăng có...,Không dùng số dòng Calendar/Reviews làm mẫu số...
3,KPI-04,Tỷ lệ bản ghi lịch còn available,Tỷ lệ bản ghi listing-ngày có trạng thái avail...,Calendar.listing_id; Calendar.date; Calendar.a...,SUM(CASE WHEN available = TRUE THEN 1 ELSE 0 E...,listing_id và date hợp lệ; available chuẩn hóa...,Số dòng listing-ngày hợp lệ có available = TRUE.,Số dòng listing-ngày hợp lệ có available = TRU...,% bản ghi listing-ngày,Một listing tại một ngày lịch và một snapshot,Tỷ lệ trạng thái lịch được đánh dấu còn khả dụ...,"FALSE chỉ là không khả dụng, không chứng minh ..."
4,KPI-05,Điểm review tổng thể trung bình,Điểm review tổng thể trung bình giữa các listi...,Listings.id; Listings.review_scores_rating,AVG(review_scores_rating) trên mỗi listing duy...,Một snapshot; điểm là số hữu hạn trong miền 1–...,Tổng điểm review tổng thể của các listing có đ...,Số listing duy nhất có điểm review tổng thể hợ...,Điểm (thang quan sát 1–5),Một listing tại snapshot,Điểm tổng thể trung bình giữa các listing có đ...,Dữ liệu snapshot hiện tại quan sát được miền 1...
5,SUP-01,Trung vị số review trên mỗi listing,Trung vị số review được ghi nhận trên mỗi list...,Listings.id; Listings.number_of_reviews,"MEDIAN(number_of_reviews), mỗi listing một lần.",Một snapshot; number_of_reviews là số hợp lệ v...,Không áp dụng.,Không áp dụng.,Review/listing,Một listing tại snapshot,Mức số lượng review điển hình của listing tron...,Không thay NULL bằng 0; không mặc định số dòng...
6,SUP-02,Số bản ghi review trong kỳ,Số bản ghi review hợp lệ được ghi nhận trong k...,Reviews.listing_id; Reviews.id; Reviews.date,"Đếm bản ghi theo khóa (listing_id, id) sau kiể...",Ngày review hợp lệ và nằm trong kỳ; listing_id...,Số bản ghi review hợp lệ trong kỳ.,Không áp dụng.,Bản ghi review,Một bản ghi review,"Lượng review quan sát được trong kỳ, có thể ph...",Không suy ra số booking; không coi review date...


## 6. Định nghĩa Technical KPI

In [7]:

technical_kpi = pd.read_csv(
    KPI_DIR / "technical_kpi_definitions.csv"
)

display(technical_kpi)


,technical_kpi_id,technical_kpi_name,definition,formula,scope,unit,grain,interpretation,caveats
0,TQ-01,Tỷ lệ thiếu dữ liệu (Missing Value Rate),Tỷ lệ ô NULL/thiếu trên tổng số ô thuộc danh s...,Tổng số ô NULL ở các trường bắt buộc / tổng số...,Báo cáo theo từng trường và tổng hợp theo nguồ...,%,Một trường dữ liệu trong một nguồn/lớp và snap...,Tỷ lệ càng thấp cho thấy mức độ đầy đủ càng ca...,Không gộp mọi cột tùy ý; phân biệt NULL hợp lệ...
1,TQ-02,Số bản ghi trùng khóa (Duplicate Key Count),Số bản ghi vượt quá một bản ghi duy nhất theo ...,COUNT(rows) - COUNT(DISTINCT business_key) hoặ...,Kiểm tra RAW hiện tại (một snapshot): Listings...,Bản ghi trùng,Theo khóa/grain của từng nguồn/lớp,Giá trị 0 nghĩa là không phát hiện bản ghi trù...,Không mặc định Reviews.id riêng là duy nhất; p...
2,TQ-03,Tỷ lệ toàn vẹn tham chiếu (Referential Integri...,Tỷ lệ bản ghi con có khóa tham chiếu tồn tại t...,Số bản ghi con có khóa cha hợp lệ / tổng bản g...,Calendar.listing_id và Reviews.listing_id tham...,%,Bản ghi cần liên kết của từng quan hệ,Tỷ lệ càng cao cho thấy liên kết giữa các nguồ...,Báo cáo riêng số khóa NULL và số khóa mồ côi; ...
3,TQ-04,Tỷ lệ bản ghi hợp lệ (Valid Record Rate),Tỷ lệ bản ghi đáp ứng toàn bộ quy tắc chất lượ...,Số bản ghi đạt tất cả kiểm tra bắt buộc / tổng...,"Kiểm tra kiểu dữ liệu, miền giá trị, trường bắ...",%,Một bản ghi tại từng nguồn/lớp,Đo tỷ lệ dữ liệu đáp ứng tiêu chí sử dụng sau ...,Công bố rõ rule và số dòng bị loại/sửa; không ...
4,TQ-05,Tính nhất quán khi chạy lại pipeline (Pipeline...,Mức độ giống nhau của số lượng bản ghi và KPI ...,So sánh row count và KPI sau lần chạy lại với ...,Thực hiện tối thiểu hai lần chạy cùng snapshot...,PASS/FAIL và chênh lệch bản ghi/KPI,Một lần chạy pipeline cho một snapshot,PASS chứng minh pipeline có thể tái hiện kết q...,Không thể tính từ file raw đơn lẻ; phải chạy p...


## 7. Kết quả tính toán Technical KPI

In [8]:

technical_results = pd.read_csv(
    OUT_DIR / "technical_kpi_results.csv"
)

display(technical_results)

print(
    "TQ-05 chưa được đánh giá vì cần bằng chứng "
    "hai lần chạy ETL với cùng dữ liệu đầu vào."
)


,technical_kpi_id,technical_kpi_name,ket_qua,don_vi,trang_thai,dien_giai
0,TQ-01,Tỷ lệ thiếu dữ liệu,0.0,%,Đã tính,"0/3,657,376 ô ở các trường bắt buộc bị thiếu."
1,TQ-02,Số bản ghi trùng khóa,0.0,Dòng trùng dư,Đã tính,Tổng số dòng trùng dư theo khóa/grain đã xác đ...
2,TQ-03,Tỷ lệ toàn vẹn tham chiếu,100.0,%,Đã tính,"1,220,158/1,220,158 khóa con có giá trị liên k..."
3,TQ-04,Tỷ lệ bản ghi hợp lệ,100.0,%,Đã tính theo luật tối thiểu công bố,"1,220,213/1,220,213 dòng đạt luật kiểm tra tối..."
4,TQ-05,Tính nhất quán khi chạy lại pipeline,NaN,PASS/FAIL,CHƯA ĐÁNH GIÁ,Cần log của ít nhất hai lần chạy ETL với cùng ...


TQ-05 chưa được đánh giá vì cần bằng chứng hai lần chạy ETL với cùng dữ liệu đầu vào.


## 8. Chi tiết kiểm tra chất lượng dữ liệu

In [9]:

technical_detail = pd.read_csv(
    OUT_DIR / "technical_kpi_detail.csv"
)

for kpi in ["TQ-01", "TQ-02", "TQ-03", "TQ-04"]:

    print("\nKết quả chi tiết:", kpi)

    display(
        technical_detail[
            technical_detail["technical_kpi_id"] == kpi
        ]
    )



Kết quả chi tiết: TQ-01


,technical_kpi_id,nguon,truong_kiem_tra,so_dong,so_o_kiem_tra,so_o_thieu,ket_qua_percent,ghi_chu,khoa_grain,so_dong_nguon,so_dong_trung_du,so_dong_thieu_thanh_phan_khoa,quan_he,tong_dong_con,khoa_con_null_or_blank,khoa_con_co_gia_tri,khoa_mo_coi,khoa_hop_le,quy_tac_hop_le,so_dong_hop_le,so_dong_khong_hop_le
0,TQ-01,Listings,"id, scrape_id",3208.0,6416.0,0.0,0.0,Chỉ tính các trường bắt buộc được khai báo tro...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,TQ-01,Calendar,"listing_id, date, available",1170950.0,3512850.0,0.0,0.0,Chỉ tính các trường bắt buộc được khai báo tro...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,TQ-01,Reviews,"listing_id, id, date",46000.0,138000.0,0.0,0.0,Chỉ tính các trường bắt buộc được khai báo tro...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,TQ-01,Neighbourhoods,"neighbourhood_group, neighbourhood",55.0,110.0,0.0,0.0,Chỉ tính các trường bắt buộc được khai báo tro...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,TQ-01,Tổng hợp,Các trường bắt buộc của 4 nguồn,NaN,3657376.0,0.0,0.0,Tỷ lệ tính trên tổng số ô thuộc các trường bắt...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Kết quả chi tiết: TQ-02


,technical_kpi_id,nguon,truong_kiem_tra,so_dong,so_o_kiem_tra,so_o_thieu,ket_qua_percent,ghi_chu,khoa_grain,so_dong_nguon,so_dong_trung_du,so_dong_thieu_thanh_phan_khoa,quan_he,tong_dong_con,khoa_con_null_or_blank,khoa_con_co_gia_tri,khoa_mo_coi,khoa_hop_le,quy_tac_hop_le,so_dong_hop_le,so_dong_khong_hop_le
5,TQ-02,Listings,NaN,NaN,NaN,NaN,NaN,"Reviews dùng khóa ghép (listing_id, id), không...",id,3208.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,TQ-02,Calendar,NaN,NaN,NaN,NaN,NaN,"Reviews dùng khóa ghép (listing_id, id), không...",listing_id + date,1170950.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,TQ-02,Reviews,NaN,NaN,NaN,NaN,NaN,"Reviews dùng khóa ghép (listing_id, id), không...",listing_id + id,46000.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,TQ-02,Neighbourhoods,NaN,NaN,NaN,NaN,NaN,"Reviews dùng khóa ghép (listing_id, id), không...",neighbourhood_group + neighbourhood,55.0,0.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Kết quả chi tiết: TQ-03


,technical_kpi_id,nguon,truong_kiem_tra,so_dong,so_o_kiem_tra,so_o_thieu,ket_qua_percent,ghi_chu,khoa_grain,so_dong_nguon,so_dong_trung_du,so_dong_thieu_thanh_phan_khoa,quan_he,tong_dong_con,khoa_con_null_or_blank,khoa_con_co_gia_tri,khoa_mo_coi,khoa_hop_le,quy_tac_hop_le,so_dong_hop_le,so_dong_khong_hop_le
9,TQ-03,NaN,NaN,NaN,NaN,NaN,100.0,Tỷ lệ tính trên khóa con có giá trị. NULL/rỗng...,NaN,NaN,NaN,NaN,Calendar.listing_id -> Listings.id,1170950.0,0.0,1170950.0,0.0,1170950.0,NaN,NaN,NaN
10,TQ-03,NaN,NaN,NaN,NaN,NaN,100.0,Tỷ lệ tính trên khóa con có giá trị. NULL/rỗng...,NaN,NaN,NaN,NaN,Reviews.listing_id -> Listings.id,46000.0,0.0,46000.0,0.0,46000.0,NaN,NaN,NaN
11,TQ-03,NaN,NaN,NaN,NaN,NaN,100.0,Tỷ lệ tính trên khóa con có giá trị. NULL/rỗng...,NaN,NaN,NaN,NaN,"Listings.(neighbourhood_group_cleansed, neighb...",3208.0,0.0,3208.0,0.0,3208.0,NaN,NaN,NaN



Kết quả chi tiết: TQ-04


,technical_kpi_id,nguon,truong_kiem_tra,so_dong,so_o_kiem_tra,so_o_thieu,ket_qua_percent,ghi_chu,khoa_grain,so_dong_nguon,so_dong_trung_du,so_dong_thieu_thanh_phan_khoa,quan_he,tong_dong_con,khoa_con_null_or_blank,khoa_con_co_gia_tri,khoa_mo_coi,khoa_hop_le,quy_tac_hop_le,so_dong_hop_le,so_dong_khong_hop_le
12,TQ-04,Listings,NaN,3208.0,NaN,NaN,100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,id và scrape_id không NULL; rating nếu có phải...,3208.0,0.0
13,TQ-04,Calendar,NaN,1170950.0,NaN,NaN,100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,listing_id có giá trị; date hợp lệ; available ...,1170950.0,0.0
14,TQ-04,Reviews,NaN,46000.0,NaN,NaN,100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"listing_id, id không NULL và date hợp lệ.",46000.0,0.0
15,TQ-04,Neighbourhoods,NaN,55.0,NaN,NaN,100.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Cả neighbourhood_group và neighbourhood đều kh...,55.0,0.0



## 9. Kết luận

- Data Dictionary mô tả các trường dữ liệu thuộc bốn nguồn.
- Mô tả nghiệp vụ được rà soát để phát hiện trường trống
  hoặc sử dụng mô tả dự phòng.
- Currency và Review Rating được kiểm tra trực tiếp
  từ dữ liệu Listings.
- TQ-01 đến TQ-04 được tính theo các quy tắc kiểm tra
  đã công bố trong chương trình.
- TQ-05 chưa thể kết luận nếu chưa có log của hai lần
  chạy pipeline ETL trên cùng snapshot.

Lưu ý: Kết quả TQ-04 phản ánh bộ quy tắc hợp lệ tối thiểu
được khai báo, không đại diện cho mọi quy tắc nghiệp vụ.
